> [!WARNING]
> **OFFLINE SUBMISSION NOTEBOOK**
> Internet is disabled. Upload fold checkpoints as a Kaggle dataset
> and point `CFG.MODEL_DIR` to it.


# B4: DINOv2-Large + GatedDepthwiseConv Fusion

**Author:** Mridankan Mandal

**Competition:** [CSIRO Image2Biomass](https://www.kaggle.com/competitions/csiro-biomass)

---

## Overview

Inference notebook for DINOv2-Large (`vit_large_patch14_dinov2.lvd142m`, 1024-d).
5-fold ensemble with GatedDepthwiseConvBlock fusion and compositional regression heads.
No metadata MLP.


## Inference and Submission Generation (B4)

In [ ]:
import os, gc, warnings
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

warnings.filterwarnings('ignore')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import glob as _glob
def _find(slug, pattern=None):
    """Find Kaggle dataset dir. If pattern given, searches subdirs too."""
    for base in [f'/kaggle/input/{slug}', *_glob.glob(f'/kaggle/input/datasets/*/{slug}')]:
        if not os.path.isdir(base): continue
        if pattern:
            if _glob.glob(os.path.join(base, pattern)): return base
            for sub in _glob.glob(os.path.join(base, '*')):
                if os.path.isdir(sub) and _glob.glob(os.path.join(sub, pattern)):
                    return sub
        return base
    raise FileNotFoundError(f"Dataset '{slug}' not found in /kaggle/input/")


In [ ]:
class CFG:
    BASE_PATH = '/kaggle/input/competitions/csiro-biomass'
    TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
    TEST_IMAGE_DIR = os.path.join(BASE_PATH, 'test')
    MODEL_DIR = _find('b4-dinov2-checkpoints', '*.pth')
    MODEL_NAME = 'vit_large_patch14_dinov2.lvd142m'
    N_FOLDS = 5
    FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
    IMG_SIZE = 518
    BATCH_SIZE = 6
    NUM_WORKERS = 0
    DROPOUT = 0.2
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {CFG.DEVICE}, Model: {CFG.MODEL_NAME}")
print(f"Checkpoints: {CFG.MODEL_DIR}")


In [ ]:
# --- Model Architecture (must match training) ---
class GatedDepthwiseConvBlock(nn.Module):
    def __init__(self, dim, kernel_size=5, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size//2, groups=dim)
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        g = torch.sigmoid(self.gate(x)); x = x * g
        x = self.dwconv(x.transpose(1,2)).transpose(1,2)
        x = self.proj(x); x = self.drop(x)
        return shortcut + x

def _make_head(nf, dropout):
    return nn.Sequential(
        nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(nf//2, 1), nn.Softplus()
    )

class BiomassModel(nn.Module):
    def __init__(self, model_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='')
        nf = self.backbone.num_features
        self.fusion = nn.Sequential(
            GatedDepthwiseConvBlock(nf, dropout=CFG.DROPOUT),
            GatedDepthwiseConvBlock(nf, dropout=CFG.DROPOUT)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head_green = _make_head(nf, CFG.DROPOUT)
        self.head_dead = _make_head(nf, CFG.DROPOUT)
        self.head_clover = _make_head(nf, CFG.DROPOUT)
    def forward(self, left, right):
        x = torch.cat([self.backbone(left), self.backbone(right)], dim=1)
        x = self.pool(self.fusion(x).transpose(1,2)).flatten(1)
        green = self.head_green(x); dead = self.head_dead(x); clover = self.head_clover(x)
        gdm = green + clover; total = gdm + dead
        return torch.cat([green, dead, clover, gdm, total], dim=1)

print("Model architecture defined")


In [ ]:
# --- Inference ---
def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])

class TestBiomassDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.paths = df['image_path'].values
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, os.path.basename(self.paths[idx]))
        img = cv2.imread(path)
        if img is None: img = np.zeros((1000,2000,3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape; mid = w//2
        left, right = img[:, :mid], img[:, mid:]
        if self.transform:
            left = self.transform(image=left)['image']
            right = self.transform(image=right)['image']
        return left, right

@torch.no_grad()
def predict_test(model, loader, device):
    model.eval(); all_preds = []
    for left, right in tqdm(loader, desc='Inference'):
        with autocast('cuda'):
            preds = model(left.to(device), right.to(device))
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)

test_long = pd.read_csv(CFG.TEST_CSV)
test_long['image_id'] = test_long['sample_id'].str.split('__').str[0]
test_df = test_long.drop_duplicates('image_id')[['image_id','image_path']].reset_index(drop=True)
print(f"Test images: {len(test_df)}")

test_dataset = TestBiomassDataset(test_df, CFG.TEST_IMAGE_DIR, get_val_transforms())
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True)

all_fold_preds = []
for fold in CFG.FOLDS_TO_TRAIN:
    ckpt_path = os.path.join(CFG.MODEL_DIR, f'fold{fold}_best.pth')
    if not os.path.exists(ckpt_path):
        print(f'Checkpoint not found: {ckpt_path}, skipping fold {fold}.')
        continue
    model = BiomassModel(CFG.MODEL_NAME, pretrained=False).to(CFG.DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=CFG.DEVICE, weights_only=True))
    print(f'Loaded fold {fold}')
    all_fold_preds.append(predict_test(model, test_loader, CFG.DEVICE))
    del model; gc.collect(); torch.cuda.empty_cache()

if not all_fold_preds:
    raise RuntimeError('No fold checkpoints found.')
avg_preds = np.mean(all_fold_preds, axis=0)
print(f'Ensemble of {len(all_fold_preds)} folds, shape: {avg_preds.shape}')


In [ ]:
# --- Submission ---
image_ids = test_df['image_id'].values
pred_map = {}
for i, img_id in enumerate(image_ids):
    for j, tn in enumerate(CFG.TARGET_COLS):
        pred_map[(img_id, tn)] = float(avg_preds[i, j])

test_long['target'] = test_long.apply(
    lambda row: pred_map.get((row['image_id'], row['target_name']), 0.0), axis=1)
df_sub = test_long[['sample_id','target']].copy()

sample_sub = pd.read_csv(os.path.join(CFG.BASE_PATH, 'sample_submission.csv'))
df_sub = sample_sub[['sample_id']].merge(df_sub, on='sample_id', how='left')
df_sub['target'] = df_sub['target'].fillna(0.0)
df_sub.to_csv('submission.csv', index=False)
print(f'Saved submission.csv, shape: {df_sub.shape}')
print(df_sub.head(10).to_string(index=False))
